#구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pwd

In [ ]:
%cd drive/MyDrive/ITLogForce_ChatBot/

# pip 설치, api 설정

In [ ]:
!pip install langchain langchain_community langchain-openai langchain_chroma

In [ ]:
!pip install unstructured

In [ ]:
!python3 -m pip install konlpy

In [ ]:
!pip install rank_bm25

In [2]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY']= userdata.get('OPENAI_API_KEY')
os.environ['LANGSMITH_ENDPOINT']= userdata.get('LANGSMITH_ENDPOINT2')
os.environ['LANGSMITH_PROJECT']= userdata.get('LANGSMITH_PROJECT2')
os.environ['LANGSMITH_TRACING']= userdata.get('LANGSMITH_TRACING')
os.environ['LANGSMITH_API_KEY']= userdata.get('LANGSMITH_API_KEY2')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# # 평가
# !pip install -U langsmith langchain-teddynote
# !pip install -qU langchain-teddynote

In [ ]:
# from dotenv import load_dotenv

# load_dotenv()

# from langchain_teddynote import logging

# logging.langsmith("dsba6_IT_ChatBot")


#문서 읽어오기, 임베딩, 벡터db
- 경로 아래에 있는 모든 txt파일.
- input_chunk_size : 청크 개당 길이
- input_chunk_overlao : 청크 오버랩
- filetype : 읽어오는 파일 타입
- len(docs) : 청크 개수
- model_name : 임베딩 모델 이름

In [ ]:
import os
from glob import glob
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader

# 한국어 형태소 분할
from konlpy.tag import Okt

okt = Okt()

def len_okt(text):
    tokens = [token for token in okt.morphs(text)]

    return len(tokens)

def okt_tokenize(text):
    return [token for token in okt.morphs(text)]

# 텍스트 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

input_chunk_size = 300
input_chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " ", ""],
    chunk_size = input_chunk_size,
    chunk_overlap = input_chunk_overlap,
    length_function = len_okt
)

In [ ]:
direct = "./Data_Files(text)"
filetype = 'txt'
print(f"디렉토리 경로'{direct}' 아래 '{filetype}'형식의 파일들을 로드합니다.\n")
loader = DirectoryLoader(direct ,glob="*."+filetype, loader_cls=TextLoader)
docs = loader.load()
print("로드 된 파일 개수 :", len(docs))

In [ ]:
print(f"청크사이즈{input_chunk_size}, 청크오버랩 사이즈{input_chunk_overlap} 인 text_splitter를 실행합니다.")

In [ ]:
texts = text_splitter.split_documents(docs)

In [ ]:
# # 랜덤 청크 확인용
# texts[5]

In [ ]:
print("텍스트 스플릿 된 총 청크 개수: ",len(texts))

In [ ]:
'''
RAG_architecture(text_splitter)의
문서 로드부터 스플릿, 임베딩 이전까지 바꿔 사용.
'''
import os
from glob import glob

# 텍스트 분할
from langchain_text_splitters import RecursiveJsonSplitter

input_max_chunk = 1000

splitter = RecursiveJsonSplitter(max_chunk_size=input_max_chunk)

# 폴더 아래에 있는 json파일 전부 다 열기
import json
import requests

# 폴더 경로 지정
filePath = './Data_Files(json)'
filetype = 'json'

# 해당 디렉토리 아래에 있는 파일들의 경로를 저장하는 함수와 리스트 생성
def list_files_recursively(directory):
    jsonFileName = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            jsonFileName.append(os.path.join(root, file))
    return jsonFileName

jsonFileName = list_files_recursively(filePath)

# 읽어 온 파일들 하나의 변수에 저장
docs = {}   #allData

for i in range(len(jsonFileName)):
    data = open(jsonFileName[i],'r', encoding='utf-8')
    json_data = json.load(data)
    docs.update(json_data)

# 문서 분할
texts = splitter.create_documents(texts = [docs],ensure_ascii = False)

# len(texts) # 문서 개수 확인

In [ ]:
# 임베딩
from langchain_community.embeddings import HuggingFaceEmbeddings

model_name = 'nlpai-lab/KURE-v1'
# model_name = 'jhgan/ko-sbert-nli'
# model_kwargs = {'device': 'cpu'} # cpu 사용하려면
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True}
print(f"hf_embedding_model설정, 모델 이름 : {model_name}\n")
hf_embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [ ]:
# 임베딩 벡터 경로 지정
# 파일 형식(txt/json)_청크-오버랩_임베딩모델
embedding_directory = './'+filetype+'_'+str(input_chunk_size)+'-'+str(input_chunk_overlap)+'_'+model_name
print(f"임베딩 벡터db 경로: {embedding_directory}")

In [ ]:
from langchain_chroma import Chroma
# save_directory 가 존재한다면 해당 경로를 db로 설정
if os.path.exists(embedding_directory):
    # 기존 DB 로드
    db = Chroma(embedding_directory=embedding_directory, embedding_function=hf_embedding_model)
    print("기존 Chroma DB를 로드했습니다.")
else:
    # 새 DB 생성 및 저장
    db = Chroma.from_documents(texts, hf_embedding_model, persist_directory=embedding_directory)
    print("새로운 Chroma DB를 생성하고 저장했습니다.")

# 검색, 프롬프트, LLM 설정
- k_num : 검색 개수
- bm_k_num : bm_retriever 의 검색 개수
- retriever1_rate : retriever의 비율(1이하)
- prompt_content : 프롬프트 내용
- LLM_model : LLM모델 (현재 gpt-4o-mini만 사용)


In [ ]:
# 문서 검색기 1
k_num = 3
retriever = db.as_retriever(
    search_kwargs = {'k': k_num}
)

# 문서 검색기 2
from langchain_community.retrievers import BM25Retriever
bm_k_num = 3
bm_retriever = BM25Retriever.from_documents(
    documents = docs,
    preprocess_func = okt_tokenize,
)

bm_retriever.k = bm_k_num

# 앙상블 retriever
from langchain.retrievers import EnsembleRetriever
retriever1_rate = 0.5
ensemble_retriever = EnsembleRetriever(
    retrievers = [retriever, bm_retriever],
    weights = [retriever1_rate, 1-retriever1_rate]
)

# 검색 문서 포맷
def format_docs(docs):
    return "\n\n".join(document.page_content for document in docs)

In [ ]:
# 프롬프트
from langchain_core.prompts import ChatPromptTemplate

prompt_content = """
            You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer the question.
            If you don't know the answer, just say that you don't know.

            Answer in Korean.

            #Context:
            {context}
            """

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            prompt_content,
        ), ("human", "{question}"),
    ]
)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

LLM_model="gpt-4o-mini"
llm = ChatOpenAI(
    model = LLM_model,
    temperature = 0,
    streaming = True,
    callbacks = [StreamingStdOutCallbackHandler()],
)

# LLM 체인 형성

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

chain = {
    "context" : ensemble_retriever | RunnableLambda(format_docs),
    "question" : RunnablePassthrough()
} | prompt | llm | StrOutputParser()

In [ ]:
question = 'QA엔지니어를 뽑는 채용공고를 알려줘.'
response = chain.invoke(question)

# 평가를 위한 데이터베이스 생성

In [ ]:
# chain 실행 후 (question(input), context, answer)반환하는 함수
def context_answer_rag_answer(inputs: dict):
    context = ensemble_retriever.invoke(inputs["question"])
    return {
        "context": "\n".join([doc.page_content for doc in context]),
        "answer": chain.invoke(inputs["question"]),
        "question": inputs["question"],
    }

## QuestionList를 for문 돌려서 정보 저장

In [ ]:
QuestionList = ["QA엔지니어를 뽑는 채용공고를 알려줘.",
                "나는 mongoDB를 할 수 있어. 해당 기술을 살리려면 어디에 취업 준비를 해야 할까?",
                "서버보안 관련 업무를 하려면 뭘 준비해야 해?",
                "실무경험이 부족한 대학생이 할 수 있는 준비는 뭐가 있을까.",
                "웹퍼블리셔가 되려면 뭘 준비해야해?",
                "나는 3년차 백엔드 개발자야. 이직을 고려하고 있는데 내가 지원서를 제출할 수 있을만한 곳의 채용공고를 알려줘.",
                "AI 직무에 필요한 기술 역량은 어떤 게 있어? 일단 나는 Java을 할 줄 알아.",
                "정보처리기사 자격증은 가지고 있지만 포트폴리오가 없는 경우 지원 가능한 직무는 뭐가 있어?",
                "난 신입 게임 서버 개발자야. 현재 근무하고 있는 회사에서 아직 1년차 채 되지 않았는데, 회사가 나랑 너무 안 맞아서 이직을 고려중이야.",
                "안드로이드 개발자 직군 중에 마감일자가 5월 20일 이전인 공고를 보여줘",
                "서울시내에서 출퇴근이 편한 곳에서 일하고싶어.",
                "인공지능/머신러닝 개발 직군 중에 육아휴직이 보장되는 기업이 있을까?",
                "복장규정이 빡세지 않은 곳에서 일할래. 나는 웹 풀스택 개발자로 근무한 경험이 5년 이상이야.",
                "수평적인 분위기가 좋아.",
                "프론트엔드 개발 하기 싫다… 근데 내가 할 줄 아는게 그것뿐이긴 해.",
                "3년차 백엔드 개발자인데, 회사돈으로 책 사서 공부하고싶다.",
                "요즘 IT업계에서 제일 인기있는 기술은 뭐야?",
                "인턴을 가장 많이 뽑는 기업은 어디야?",
                "1. 개발 할 때 어떤 프로그램을 사용하나요?",
                "2. 전공생이나 취업 준비생들에게 추천하는 활동은 무엇이 있을까?",
                "3. 직무를 수행하는데 필요한 지식은 어떤 것이 있나요? 혹은 갖추고 있어야하는 지식이 있나요?",
                "4. 직무의 전망에 대해서 알려줘",
                "5. 직무를 수행하면서 노하우가 있을까요?",
                "6. 직무를 결정하지 못했는데 조언해줘.",
                "7. 하루 일과가 궁금합니다.",
                "8. 회사의 조직 구성이 어떻게 되나요?",
                "9. 회사 내의 복리후생이 무엇이 있나요?",
                "10. 회사 내의 특별한 사내 문화(분위기)가 있나요?",
]

In [ ]:
inputs = []
context = []
outputs = []
for i in range(len(QuestionList)):
    result = context_answer_rag_answer({"question": QuestionList[i]})
    inputs.append(result["question"])
    context.append(result["context"])
    outputs.append(result["answer"])

In [ ]:
qa_pairs = [{"question": q, "answer": a, "context":c} for q, a, c in zip(inputs, outputs, context)]
df = pd.DataFrame(qa_pairs)

In [ ]:
# 반환되는 값으로 df 생성
import pandas as pd
from langsmith import Client

client = Client()
dataset_name = "RAG_TEST_DATASET"


# 데이터셋 생성 함수
def create_dataset(client, dataset_name, description=None):
    for dataset in client.list_datasets():
        if dataset.name == dataset_name:
            return dataset

    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description=description,
    )
    return dataset

dataset = create_dataset(client, dataset_name)
# 생성된 데이터셋에 예제 추가:langchain 밖의 df에 각각의 요소값을 더해야 할 때
client.create_examples(
    inputs=[{"question": q} for q in df["question"].tolist()],
    context = [{"context": q} for q in df["context"].tolist()],
    outputs=[{"answer": a} for a in df["answer"].tolist()],
    dataset_id=dataset.id,
)

# 평가자 생성

In [ ]:
# 평가자 프롬프트
from langchain import hub

llm_evaluator_prompt = hub.pull("teddynote/context-answer-evaluator")
# llm_evaluator_prompt.pretty_print()

# 평가자 생성
custom_llm_evaluator = (
    llm_evaluator_prompt
    | ChatOpenAI(temperature=0.0, model="gpt-4o-mini")
    | StrOutputParser()
)

In [ ]:
# 커스텀 평가자
from langsmith.schemas import Run, Example


def custom_evaluator(run: Run, example: Example) -> dict:
    # LLM 생성 답변, 정답 답변 가져오기
    llm_answer = run.outputs.get("answer", "")
    context = run.outputs.get("context", "")
    question = example.outputs.get("question", "")

    # 랜덤 점수 반환
    score = custom_llm_evaluator.invoke(
        {"question": question, "answer": llm_answer, "context": context}
    )
    return {"key": "custom_score", "score": float(score)}

# 평가 진행

In [ ]:
from langsmith.evaluation import evaluate

# 데이터셋 이름 설정
dataset_name = "RAG_TEST_DATASET"

# 실행
experiment_results = evaluate(
    context_answer_rag_answer,
    data=dataset_name,
    evaluators=[custom_evaluator],
    experiment_prefix="CUSTOM-LLM-EVAL",
    # 실험 메타데이터 지정
    metadata={
        "variant": "Custom LLM Evaluator 활용한 평가",
    },
)